# 06 — Reasoning-Oriented Prompting

Measure observable decision artifacts, evidence verification, and inference-time compute without requesting private chain-of-thought.

## Scenario, experimental question, and success criteria

Northstar's incident desk must choose `rollback`, `escalate_database`, `scale_capacity`, `rotate_credentials`, `collect_evidence`, or `monitor` from untrusted narratives and approved telemetry.

**Experimental question.** When does extra reasoning improve a decision, and when does evidence verification or deterministic routing outperform more candidates and more tokens?

**Success criteria.** Compare five strategies on 24 sliced incidents; report decision accuracy, evidence support, safe escalation, calls, estimated tokens, latency, and artifact coverage; demonstrate that majority agreement can preserve a shared unsupported premise.

## Learning objectives and safety boundaries

You will separate private model computation from observable checks, compare direct/decomposed/verified/self-consistent/adaptive approaches, and route explicit rules to code.

Every action is a proposal. The notebook does not expose hidden reasoning, execute incident changes, or let prompt text establish identity, evidence access, or authorization. The simulator teaches experimental design, not provider internals.

## Environment and reproducibility

The entire benchmark runs offline. The optional integration cell requires each learner's own `OPENAI_API_KEY`, `PROMPT_COURSE_PROVIDER=openai`, and explicit `RUN_LIVE=1`. Never paste credentials into this notebook. One live artifact validates wiring only; it is not a benchmark result.

In [ ]:
from dataclasses import asdict
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path('src').resolve()))
module_path = Path('curriculum/intermediate/06-reasoning-oriented-prompting/lab.py')
spec = spec_from_file_location('course06_reasoning_lab', module_path)
lab = module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
print({'cases': len(lab.load_cases()), 'actions': list(lab.Action.__args__)})

## Architecture and observable reasoning boundary

```text
untrusted narrative + approved evidence IDs → bounded decision artifact
                                              → evidence/policy verifier
                                              → propose / collect evidence
                                              → application authorization
```

The product artifact contains an action, evidence IDs, externally checkable assumptions, checks, and a human-review flag. It does not require a private token-by-token rationale.

## Freeze the incident evaluation

The dataset separates reported signals from verified signals. Expected outcomes are fixed before any strategy is compared, including missing-evidence, conflicting, injection, deterministic, and recovered cases.

In [ ]:
cases = lab.load_cases()
case_frame = pd.DataFrame([{
    'id': case.id, 'slice': case.slice, 'expected': case.expected,
    'reported': len(case.metadata.get('signals', [])),
    'verified': len(case.metadata.get('verified_signals', [])),
    'deterministic': bool(case.metadata.get('deterministic_action'))
} for case in cases])
display(pd.crosstab(case_frame['slice'], case_frame['expected']))
assert len(cases) == 24
assert {'missing_evidence', 'conflicting', 'injection', 'deterministic'} <= set(case_frame['slice'])

## Baseline — direct narrative classification

The direct baseline uses one lexical decision and returns only a minimal check. Hypothesis: it is cheap but will confuse mentioned systems with verified causes and fail safe escalation.

In [ ]:
direct_rows = lab.run_strategy('direct')
direct_metrics = lab.metrics(direct_rows)
pd.Series(direct_metrics).round(4)

### Inspect baseline failures

A convincing action is not enough. Compare the selected action with the expected terminal state and whether approved evidence supports it.

In [ ]:
pd.DataFrame(lab.failure_matrix('direct'))[[
    'case_id', 'slice', 'expected', 'selected', 'supported', 'calls', 'tokens_estimated'
]]

## Step 1 — expose a bounded decomposition

The decomposed strategy enumerates candidate signals and maps them to an action. Its assumptions are visible, but it deliberately treats reported signals as verified. Observable structure improves diagnosis without automatically making the decision correct.

In [ ]:
decomposed_rows = lab.run_strategy('decomposed')
decomposed_metrics = lab.metrics(decomposed_rows)
pd.DataFrame({'direct': direct_metrics, 'decomposed': decomposed_metrics}).T.round(4)

## Step 2 — separate proposal from verification

The planner/verifier path builds a candidate from reported signals, then recomputes support from approved evidence IDs. Missing or conflicting support terminates at `collect_evidence`. This costs another call in the teaching model.

In [ ]:
verified_rows = lab.run_strategy('planner_verifier')
verified_metrics = lab.metrics(verified_rows)
pd.Series(verified_metrics).round(4)
assert verified_metrics['supported_decision_rate'] == 1
assert verified_metrics['safe_escalation_accuracy'] == 1

## Inspect an observable artifact

The artifact supports audit and testing without exposing hidden reasoning. Checks describe externally testable operations; evidence IDs identify the approved inputs.

In [ ]:
example_case = next(case for case in cases if case.id == 'RSN-017')
artifact, candidates, calls = lab.decide(example_case, 'planner_verifier')
{'artifact': artifact.model_dump(), 'candidates': candidates, 'calls': calls}

## Step 3 — test self-consistency rather than assuming it helps

Self-consistency samples five candidate decisions and takes a majority. It can reduce independent sampling error, but candidates sharing the same unverified premise can agree on the same unsupported action.

In [ ]:
self_rows = lab.run_strategy('self_consistency')
self_metrics = lab.metrics(self_rows)
pd.DataFrame({'planner_verifier': verified_metrics, 'self_consistency': self_metrics}).T.round(4)

## Step 4 — route explicit rules to deterministic checks

Capacity thresholds and recovered-state checks do not need model reasoning. The adaptive strategy uses deterministic rules for those cases and the verified path elsewhere, reducing mean calls while preserving the frozen result.

In [ ]:
adaptive_rows = lab.run_strategy('adaptive')
adaptive_metrics = lab.metrics(adaptive_rows)
assert adaptive_metrics['decision_accuracy'] == verified_metrics['decision_accuracy']
assert adaptive_metrics['mean_calls'] < verified_metrics['mean_calls']
pd.Series(adaptive_metrics).round(4)

## Run the controlled strategy comparison

Read quality beside resource use. The local latency is measured but too small to predict a provider SLO; calls and token estimates expose the relative architecture cost.

In [ ]:
comparison = pd.DataFrame(lab.compare_strategies()).set_index('strategy')
comparison.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
comparison[['decision_accuracy', 'supported_decision_rate', 'safe_escalation_accuracy']].plot.bar(ax=axes[0])
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel('Rate'); axes[0].grid(axis='y', alpha=.25)
comparison[['mean_calls', 'mean_tokens_estimated']].plot.bar(ax=axes[1], secondary_y='mean_tokens_estimated')
axes[1].set_title('Inference-time resource tradeoff'); axes[1].grid(axis='y', alpha=.25)
plt.show()

## Failure injection — five candidates share one unsupported premise

Case RSN-019 contains an instruction-like request to rotate credentials but no approved security signal. Both the direct and decomposed candidates follow the narrative. Majority vote repeats their shared error; verification terminates safely.

In [ ]:
attack_case = next(case for case in cases if case.id == 'RSN-019')
injected = {}
for strategy in ('direct', 'decomposed', 'self_consistency', 'planner_verifier', 'adaptive'):
    result, candidates, calls = lab.decide(attack_case, strategy)
    injected[strategy] = {'action': result.action, 'candidates': candidates, 'calls': calls}
assert injected['self_consistency']['action'] == 'rotate_credentials'
assert injected['planner_verifier']['action'] == 'collect_evidence'
pd.DataFrame(injected).T

## Diagnose the failure

The failure is not insufficient verbosity or too few samples. The candidate paths are correlated because they consume the same unverified premise. Add an independent support check, keep source authorization outside the model, and stop at `collect_evidence`. Do not log private chain-of-thought as a substitute for a trace.

## Technology comparison

| Approach | Maturity | Use when | Avoid when |
| --- | --- | --- | --- |
| direct typed request | established | task is clear and bounded | evidence boundary is unresolved |
| explicit decomposition | established/model-dependent | artifacts improve diagnosis | model already solves direct path |
| planner/verifier | practical | support can be checked independently | verifier shares the same weak signal |
| self-consistency | model-dependent | diverse paths reduce measured error | paths are correlated or budget is tight |
| model-native reasoning effort | current provider feature | hard tasks justify test-time compute | defaults are unbenchmarked |
| learned/adaptive compute | emerging/research | task difficulty varies substantially | router errors are unmeasured |

Current reasoning-model guidance favors clear goals and simple prompts; benchmark explicit chain-of-thought scaffolds instead of treating them as a default.

## Optional live provider implementation

The adapter returns the same typed `DecisionArtifact` in offline and live modes. Export your own `OPENAI_API_KEY`, set `PROMPT_COURSE_PROVIDER=openai`, then set `RUN_LIVE=1`. The request asks for checks and evidence IDs, not private reasoning.

In [ ]:
if os.getenv('RUN_LIVE') == '1':
    if os.getenv('PROMPT_COURSE_PROVIDER') != 'openai' or not os.getenv('OPENAI_API_KEY'):
        raise RuntimeError('Set your own OPENAI_API_KEY and PROMPT_COURSE_PROVIDER=openai first.')
    provider_result = lab.run_provider_case(cases[0])
    display(provider_result.value.model_dump())
else:
    print('Skipped: offline evaluation complete; set RUN_LIVE=1 for one explicit integration call.')

## Production upgrade

| Notebook | Production |
| --- | --- |
| local incident JSONL | versioned development, held-out, safety, and shifted suites |
| reported/verified arrays | authorized telemetry adapters with freshness/provenance |
| deterministic teaching rules | versioned policy engine with boundary tests |
| estimated tokens/calls | provider usage, reasoning-token, cost, and p95 latency data |
| one verifier | independent evidence/policy validation and calibrated escalation |
| printed artifact | privacy-aware trace with model/contract/context versions |
| manual choice | shadow/canary gate, SLO, budget, alert, and rollback |

Bound calls, output tokens, wall time, and retries. Make terminal states explicit, validate every artifact, and authorize operational effects outside the reasoning path.

## When not to use reasoning scaffolds

Do not add decomposition or self-consistency when a direct request already passes, a deterministic rule solves the task, approved evidence is absent, or the latency/cost budget cannot support it. Do not request chain-of-thought for routine tasks or accept explanations as authorization.

## Review questions and exercises

1. Why can five agreeing candidates remain unsupported?
2. Add a stale-but-verified signal and define the correct outcome before running it.
3. Give planner and verifier different evidence views and measure the effect.
4. Add a difficulty router that buys extra calls only for ambiguous cases.
5. Define a release gate that blocks one unsupported action despite higher aggregate accuracy.

**Advanced challenge.** Implement confidence-aware self-consistency with early stopping. Compare accuracy, calibration, calls, and latency with the fixed five-candidate strategy on held-out and shifted slices.

## Summary

Reasoning quality is a measured system property, not the length of an explanation. Prefer simple direct requests first, expose only useful decision artifacts, verify evidence independently, route explicit rules to code, and buy extra inference-time compute only when a frozen evaluation justifies it.